# Scene Occlusion

In [ ]:
import numpy as np
import open3d as o3d
from pathlib import Path
import trimesh
import sys
sys.path.insert(0, '../')
import drm
import copy

DATASET_DIR = Path(r"../../../datasets/V-Scan/data")   # <-- change this
FOLDER_NAME = "bedroom_Leica-P30_1775809921180"
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"
VOXEL_SIZE = 0.05
# Distance threshold: the max coverage distance (metres)
THRESHOLD_RESOLUTION = 0.1

%load_ext autoreload
%autoreload 2


In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / FOLDER_NAME / REFERENCE_NAME
empty_scene_path = dataset_dir / FOLDER_NAME / EMPTY_SCENE_NAME
var_paths = sorted((dataset_dir/FOLDER_NAME).glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)
ref_scan_pos = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]

varPcds = []
varPosses = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
    varPcd = varPcd.voxel_down_sample(VOXEL_SIZE)
    posTransform = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
    varPcds.append(varPcd)
    varPosses.append(posTransform)

In [ ]:
movedVarPcd = copy.deepcopy(varPcds[0]).translate(-varPosses[0])
newScene = drm.visualise_open3d(movedVarPcd)
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
newScene.add_geometry(drm.visualise_open3d(movedVarPcd.get_minimal_oriented_bounding_box(), random_color=True))
newScene.show()

In [ ]:
def build_occlusion_grid(
    reference: o3d.geometry.PointCloud,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
) -> tuple[o3d.geometry.VoxelGrid, o3d.geometry.VoxelGrid]:
    """
    Builds occupied and occluded voxel grids from a reference point cloud.

    Internally shifts the cloud so the scanner is at the origin, rotates into
    the OBB local frame for compact voxelization, then ray marches from the
    scanner through each occupied voxel to find the occluded shadow behind it.
    Both grids are returned in world space.

    Parameters
    ----------
    reference   : point cloud that defines the geometry (e.g. var_pcd)
    scanner_pos : world-space position of the scanner that captured reference
    voxel_size  : edge length of each voxel in metres

    Returns
    -------
    occupied_grid : VoxelGrid — voxels containing at least one point
    occluded_grid : VoxelGrid — empty voxels in the shadow behind geometry
    """
    pts_shifted = np.asarray(reference.points) - scanner_pos

    pts_shifted = np.asarray(reference.points) - scanner_pos

    shifted_pcd        = o3d.geometry.PointCloud()
    shifted_pcd.points = o3d.utility.Vector3dVector(pts_shifted)
    obb    = shifted_pcd.get_minimal_oriented_bounding_box()
    R      = np.asarray(obb.R)
    center = np.asarray(obb.center)

    pts_local = (pts_shifted - center) @ R
    min_bound = pts_local.min(axis=0)
    max_bound = pts_local.max(axis=0)
    grid_size = np.floor((max_bound - min_bound) / voxel_size).astype(int) + 1

    voxel_indices = np.floor((pts_local - min_bound) / voxel_size).astype(int)
    occupied_set  = set(map(tuple, voxel_indices))

    # FIX: split into two steps to avoid precedence bug
    origin_local = (np.zeros(3) - center) @ R
    origin_voxel = (origin_local - min_bound) / voxel_size

    occluded_set = set()
    for voxel in occupied_set:
        ray_dir    = np.array(voxel, dtype=float) + 0.5 - origin_voxel
        ray_length = np.linalg.norm(ray_dir)
        if ray_length == 0:
            continue
        ray_dir_n = ray_dir / ray_length
        t         = ray_length + 1.0
        t_max     = ray_length + np.linalg.norm(grid_size)

        while t < t_max:
            current = np.floor(origin_voxel + t * ray_dir_n).astype(int)
            if np.any(current < 0) or np.any(current >= grid_size):
                break
            current_tuple = tuple(current)
            if current_tuple not in occupied_set:
                occluded_set.add(current_tuple)
            t += 1.0

    def set_to_voxel_grid(voxel_set: set, color: list) -> o3d.geometry.VoxelGrid:
        indices       = np.array(list(voxel_set))
        centres_world = (indices + 0.5) * voxel_size + min_bound
        centres_world = centres_world @ R.T + center + scanner_pos
        pcd           = o3d.geometry.PointCloud()
        pcd.points    = o3d.utility.Vector3dVector(centres_world)
        pcd.paint_uniform_color(color)
        return o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size)

    occupied_grid = set_to_voxel_grid(occupied_set, [0.86, 0.24, 0.24])
    occluded_grid = set_to_voxel_grid(occluded_set, [0.24, 0.47, 0.86])

    return occupied_grid, occluded_grid


def get_invisible_points_grid(
    points: o3d.geometry.PointCloud,
    reference: o3d.geometry.PointCloud,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
) -> tuple[o3d.geometry.PointCloud, o3d.geometry.PointCloud]:
    """
    Classifies each point in `points` as invisible (occluded or inside geometry)
    or visible, using the occlusion grid built from `reference`.

    Parameters
    ----------
    points      : point cloud to classify (e.g. uncovered candidate points)
    reference   : point cloud that defines the geometry (e.g. var_pcd)
    scanner_pos : world-space position of the scanner that captured reference
    voxel_size  : must match the value used to build the grid

    Returns
    -------
    invisible : points inside occupied or occluded voxels
    visible   : remaining points
    """
    occupied_grid, occluded_grid = build_occlusion_grid(reference, scanner_pos, voxel_size)

    pts_world      = o3d.utility.Vector3dVector(np.asarray(points.points))
    in_occupied    = np.asarray(occupied_grid.check_if_included(pts_world))
    in_occluded    = np.asarray(occluded_grid.check_if_included(pts_world))
    invisible_mask = in_occupied | in_occluded

    invisible = points.select_by_index(np.where(invisible_mask)[0])
    visible   = points.select_by_index(np.where(~invisible_mask)[0])
    return invisible, visible


def visualise_occlusion_grid(
    occupied_grid: o3d.geometry.VoxelGrid,
    occluded_grid: o3d.geometry.VoxelGrid,
    scanner_pos: np.ndarray,
    voxel_size: float = 0.05,
    show_occupied: bool = True,
    show_occluded: bool = True,
    show_scanner: bool = True,
) -> list[o3d.geometry.Geometry]:
    """
    Returns a list of native o3d geometries ready for show_geometries() or
    o3d.visualization.draw_geometries().

    Parameters
    ----------
    occupied_grid : first return value of build_occlusion_grid
    occluded_grid : second return value of build_occlusion_grid
    scanner_pos   : world-space scanner position, used to place the marker sphere
    voxel_size    : used to size the scanner marker sphere
    """
    geometries = []
    if show_occupied:
        geometries.append(occupied_grid)
    if show_occluded:
        geometries.append(occluded_grid)
    if show_scanner:
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=voxel_size * 1.5)
        sphere.translate(scanner_pos)
        sphere.paint_uniform_color([1.0, 0.85, 0.0])
        sphere.compute_vertex_normals()
        geometries.append(sphere)
    return geometries

In [ ]:
occupied_grid, occluded_grid = build_occlusion_grid(movedVarPcd, [0,0,0], voxel_size=0.1)

#invisible, visible = get_invisible_points_grid(uncovered_points, var_pcd, scanner_pos)


In [ ]:
geom = visualise_occlusion_grid(occupied_grid, occluded_grid, [0,0,0], voxel_size=0.1)
drm.visualise_open3d(geom).show()

In [ ]:
occupied_grid, occluded_grid = build_occlusion_grid(varPcds[0], varPosses[0], voxel_size=0.1)


In [ ]:
geom = visualise_occlusion_grid(occupied_grid, occluded_grid, varPosses[0], voxel_size=0.1)
drm.visualise_open3d(geom).show()

In [ ]:

# Optionally overlay the point cloud
cloud = drm.o3d_pointcloud_to_trimesh(varPcds[0])
scene.add_geometry(cloud, node_name="cloud")

scene.show()

## Experiments